# Boosting in Deep Learning

This notebook demonstrates the idea of boosting using a practical **AdaBoost-style ensemble of neural networks**. Each neural network is trained sequentially, with more emphasis placed on samples that previous models classified incorrectly.

**Dataset:** MNIST handwritten digits

**Goal:** Compare a single neural network with a boosted ensemble.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import accuracy_score, classification_report

np.random.seed(42)
tf.random.set_seed(42)

print('TensorFlow version:', tf.__version__)

## 1. Load and preprocess MNIST

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Flatten 28x28 images into 784 features
x_train = x_train.reshape(-1, 784)
x_test = x_test.reshape(-1, 784)

print('Training data:', x_train.shape)
print('Test data:', x_test.shape)

## 2. Create a neural-network weak learner

The learner is intentionally small so that several models can be combined into an ensemble.

In [ ]:
def create_model():
    model = keras.Sequential([
        layers.Input(shape=(784,)),
        layers.Dense(64, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

## 3. Train the boosted neural-network ensemble

For each round:
1. Train a neural network using the current sample weights.
2. Calculate the weighted classification error.
3. Give the learner a higher ensemble weight when its error is lower.
4. Increase the importance of incorrectly classified samples.

This is an educational implementation of the boosting idea rather than a replacement for specialized libraries such as AdaBoost.

In [ ]:
# Use a subset to keep the notebook reasonably fast on CPU.
N = 12000
x = x_train[:N]
y = y_train[:N]

sample_weights = np.ones(N, dtype=np.float64) / N
models = []
model_weights = []
errors = []

n_estimators = 5

for t in range(n_estimators):
    print(f'\nBoosting round {t + 1}/{n_estimators}')

    model = create_model()
    model.fit(
        x, y,
        sample_weight=sample_weights,
        epochs=3,
        batch_size=128,
        validation_split=0.1,
        verbose=0
    )

    probabilities = model.predict(x, batch_size=256, verbose=0)
    predictions = np.argmax(probabilities, axis=1)
    incorrect = (predictions != y)

    weighted_error = np.sum(sample_weights * incorrect) / np.sum(sample_weights)
    weighted_error = np.clip(weighted_error, 1e-6, 1 - 1e-6)

    # SAMME-style learner weight for multiclass boosting
    alpha = np.log((1 - weighted_error) / weighted_error) + np.log(9)
    alpha = max(alpha, 0.0)

    # Increase weights for incorrectly classified samples
    sample_weights *= np.exp(alpha * incorrect)
    sample_weights /= np.sum(sample_weights)

    models.append(model)
    model_weights.append(alpha)
    errors.append(weighted_error)

    print(f'Weighted error: {weighted_error:.4f}')
    print(f'Learner weight: {alpha:.4f}')

## 4. Make ensemble predictions

In [ ]:
def boosted_predict(models, model_weights, X):
    # Weighted voting based on each model's predicted class.
    n_classes = 10
    scores = np.zeros((len(X), n_classes), dtype=np.float64)

    for model, alpha in zip(models, model_weights):
        preds = np.argmax(model.predict(X, batch_size=256, verbose=0), axis=1)
        scores[np.arange(len(X)), preds] += alpha

    return np.argmax(scores, axis=1)

ensemble_predictions = boosted_predict(models, model_weights, x_test)
ensemble_accuracy = accuracy_score(y_test, ensemble_predictions)

print(f'Boosted ensemble accuracy: {ensemble_accuracy:.4f}')
print('\nClassification Report:\n')
print(classification_report(y_test, ensemble_predictions))

## 5. Compare with a single neural network

In [ ]:
single_model = create_model()
single_model.fit(
    x_train[:N], y_train[:N],
    epochs=3,
    batch_size=128,
    validation_split=0.1,
    verbose=0
)

single_predictions = np.argmax(single_model.predict(x_test, batch_size=256, verbose=0), axis=1)
single_accuracy = accuracy_score(y_test, single_predictions)

print(f'Single neural network accuracy: {single_accuracy:.4f}')
print(f'Boosted ensemble accuracy:      {ensemble_accuracy:.4f}')
print(f'Accuracy difference:             {ensemble_accuracy - single_accuracy:+.4f}')

## 6. Important notes

- Boosting is traditionally associated with models such as decision trees, but the boosting concept can also be applied to neural-network learners.
- In modern deep learning, **bagging, ensembles, stacking, knowledge distillation, and gradient boosting on learned features** are often more practical than repeatedly training neural networks with classical AdaBoost.
- Results vary because neural-network training is stochastic and depend on the number of estimators, epochs, learning rate, and dataset size.
- For a production implementation, consider stronger architectures, early stopping, learning-rate scheduling, and a carefully designed multiclass boosting algorithm.